In [1]:
from simulation import citygraph_dataset
# from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from learning.nsgaii import main as main_ngsaii
from learning.simulated_annealing import main as main_sa
# from learning.bagloee import main as main_bagloo
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

In [2]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
import os

cfg_dir = os.path.abspath("../TNDP_learning/cfg")

### NSGAII

In [3]:
dataset_name = 'mandl'
demand_time_weight = 0.33
route_time_weight = 0.33
median_connectivity_weight = 0.33
experiment_name = f'exp_weighted_connectivity_{dataset_name}_pp_{demand_time_weight}_op_{route_time_weight}_cp_{median_connectivity_weight}'
initial_routes_name = experiment_name+ '_starting'
generated_routes_name = experiment_name + '_generated'

In [4]:
with initialize_config_dir(config_dir=cfg_dir, version_base=None):
    
    cfg_eval = compose(
        config_name="nsgaii",   # из @hydra.main
        overrides=[
            f"+eval={dataset_name}", # конфиг в котором задается кол-во маршрутов и их макс и мин длины (можно заменить на vo для теста по ваське)
            "+model_1.weights=../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt", # путь к весам модели
            "+model_2.weights=../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt", # путь к весам модели
            f"++run_name={initial_routes_name}", # имя запуска, вернет pickle файл с тензором в output_routes
            f"++experiment.cost_function.kwargs.demand_time_weight={demand_time_weight}",
            f"++experiment.cost_function.kwargs.route_time_weight={route_time_weight}",
            f"++experiment.cost_function.kwargs.median_connectivity_weight={median_connectivity_weight}"
        ]
    )

In [5]:
# print(OmegaConf.to_yaml(cfg_eval)) # чисто проверка как выглядит конфиг 

In [ ]:
pareto_networks, pareto_costs = main_ngsaii(cfg_eval)